# PoC интеграции SRA Native LLM (TinyLlama)

Этот ноутбук представляет собой экспериментальную версию концепции (PoC), которая изначально интегрирует маршрутизатор SRA с существующим LLM (здесь TinyLlama, облегченный вариант архитектуры Llama) путем сопоставления маршрутизатора со скрытым измерением LLM.
Он предназначен для работы в Google Colab (например, на бесплатном уровне графического процессора T4).

In [ ]:
# Environment setup (Google Colab etc.)
!pip install -q transformers torch numpy

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForCausalLM

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load TinyLlama (lightweight Llama-architecture model, ~1.1B parameters)
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
print(f"Loading {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
base_llm = AutoModelForCausalLM.from_pretrained(model_id).to(device)

# Freeze the LLM weights (so that only the SRA router is trained)
for param in base_llm.parameters():
    param.requires_grad = False

# Get the hidden dimension of TinyLlama (typically 2048)
hidden_size = base_llm.config.hidden_size
print(f"LLM Hidden Size (SRA Native Dimension): {hidden_size}")

## Определение маршрутизатора SRA и внешних модулей (синапсов)

Мы строим базовую систему SRA «той же размерности, что и LLM». Это устраняет необходимость в каком-либо проекционном слое.

In [ ]:
class SRARouter(nn.Module):
    def __init__(self, d_model, num_experts=2):
        super().__init__()
        # Extremely lightweight routing layer (its cost is nearly zero compared to training the LLM itself)
        self.gate = nn.Linear(d_model, num_experts)
        
    def forward(self, x):
        # x shape: [batch, seq_len, d_model]
        routing_logits = self.gate(x)
        routing_probs = torch.softmax(routing_logits, dim=-1)
        return routing_probs

class DummyCalculatorSynapse(nn.Module):
    """
    Mock of a non-trainable external module (e.g., a calculator).
    For native integration, it accepts and returns tensors of the LLM's d_model.
    """
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        # In a real system, this is where text would be passed to a calculator and the result vectorized
        
    def forward(self, x):
        # As a mock, simply return the tensor multiplied by 0.5 (a stand-in for some real processing)
        print("  [Calculator Synapse] Activated for calculation!")
        return x * 0.5

## Проверка прямого прохода встроенной интеграции

Мы векторизуем входной текст с помощью токенизатора LLM, затем моделируем отправку токенов маршрутизатора между «телом LLM» и «калькулятором».

In [ ]:
# 1. Initialize the SRA system
router = SRARouter(d_model=hidden_size, num_experts=2).to(device=device, dtype=base_llm.dtype)
calculator_synapse = DummyCalculatorSynapse(d_model=hidden_size).to(device=device, dtype=base_llm.dtype)

# 2. Prepare input data
text = "What is the result of 15 * 24 ?"
inputs = tokenizer(text, return_tensors="pt").to(device)

print(f"Input Text: '{text}'")
print(f"Token IDs shape: {inputs.input_ids.shape}")

# 3. Convert to native vectors via the LLM's embedding layer
with torch.no_grad():
    # Pass through TinyLlama's embedding layer
    embeddings = base_llm.model.embed_tokens(inputs.input_ids)
print(f"Embeddings shape (Native): {embeddings.shape}")

# 4. Run routing
# We can hand the embeddings directly to the router with no projection layer!
routing_probs = router(embeddings)
print(f"Routing Probabilities shape: {routing_probs.shape}")

# 5. Branch the processing based on the routing result (simulation)
# Per token, decide whether to dispatch to the LLM (Expert 0) or to the Calculator (Expert 1)
# Here, for simplicity, we look at the routing probabilities for the last token ('?')
last_token_probs = routing_probs[0, -1, :]
print(f"Routing Probs for last token: LLM={last_token_probs[0]:.4f}, Calc={last_token_probs[1]:.4f}")

# In a real SRA setup, the processing would be branched and merged according to these probabilities
print("\n--- Routing Execution ---")
if last_token_probs[1] > 0.5:
    # When routed to the calculator (no projection needed -- can be passed directly!)
    out_tensor = calculator_synapse(embeddings)
else:
    # When routed to the LLM
    print("  [LLM Synapse] Processing via Frozen LLM Layers...")
    # Would normally go through the Transformer blocks
    out_tensor = embeddings 

print(f"Output Tensor shape: {out_tensor.shape}")
print("\nNative integration (seamless connection without dimension conversion) verified!")

## Обучение маршрутизатора

Мы демонстрируем реальную ценность SRA: «чрезвычайно легкая тренировка».
Обучается только маршрутизатор (только параметры `d_model x 2`), а веса LLM остаются замороженными.
Маршрутизатор учится направлять арифметические задачи в калькулятор, а все остальное — в LLM.

In [ ]:
import torch.optim as optim

# 1. Prepare a toy dataset
train_data = [
    # Calculation problems (label: 1 -> Calculator)
    ("What is 5 + 5?", 1),
    ("Calculate 12 * 3", 1),
    ("15 divided by 3 is", 1),
    ("Give me the result of 100 - 42", 1),
    
    # Everyday conversation / general knowledge (label: 0 -> LLM)
    ("Hello, how are you?", 0),
    ("Tell me a story about a brave knight.", 0),
    ("What is the capital of France?", 0),
    ("Write a poem about the sea.", 0)
]

# 2. Define the loss function and optimizer
criterion = nn.CrossEntropyLoss()
# Optimize ONLY the router's weights (router.parameters())
optimizer = optim.Adam(router.parameters(), lr=0.01)

print("Training setup complete. Ready to train!")

In [ ]:
# 3. Training loop
epochs = 30

print("Starting training...")
for epoch in range(epochs):
    total_loss = 0.0
    
    for text, label in train_data:
        optimizer.zero_grad()
        
        # Vectorize the input
        inputs = tokenizer(text, return_tensors="pt").to(device)
        with torch.no_grad():
            # The LLM layers do not backpropagate (Frozen)
            embeddings = base_llm.model.embed_tokens(inputs.input_ids)
            
        # Get logits from the router (raw values before Softmax)
        # .gate is a direct Linear layer, so it returns logits
        routing_logits = router.gate(embeddings) # shape: [1, seq_len, 2]
        
        # Decide using the mean over the whole sequence
        mean_logits = routing_logits.mean(dim=1) # shape: [1, 2]
        
        # Prepare the ground-truth label
        target = torch.tensor([label]).to(device)
        
        # Compute the loss and update the weights
        loss = criterion(mean_logits, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch + 1}/{epochs} | Loss: {total_loss / len(train_data):.4f}")

print("Training finished! Router is now smart.")

## Проверка вывода после обучения

Мы проверяем, научился ли маршрутизатор, который стартовал примерно с вероятностью 50:50, понимать смысл входного текста и направлять его в правильный синапс (модуль).

In [ ]:
def test_routing(text):
    inputs = tokenizer(text, return_tensors="pt").to(device)
    with torch.no_grad():
        embeddings = base_llm.model.embed_tokens(inputs.input_ids)
        routing_probs = router(embeddings)
        
        # As during training, look at the sequence-averaged probability
        mean_probs = routing_probs.mean(dim=1)[0]
        
    print(f"Input: '{text}'")
    print(f" -> Routing: LLM={mean_probs[0]:.4f} | Calc={mean_probs[1]:.4f}")
    
    if mean_probs[1] > 0.5:
        print("    => Routed to Calculator Synapse!")
    else:
        print("    => Routed to Frozen LLM Synapse!")
    print("-" * 40)

# Run the tests
print("\n=== Post-Training Routing Test ===\n")

# 1. Arithmetic problems (unseen text)
test_routing("What is the result of 15 * 24 ?")
test_routing("Can you compute 99 / 3 for me?")

# 2. General conversation (unseen text)
test_routing("What is the meaning of life?")
test_routing("Explain quantum physics in simple terms.")